In [ ]:
import numpy as np
import sisl
import matplotlib.pyplot as plt
from ase.visualize import view
from sisl import Hamiltonian
from tqdm.auto import tqdm

from scipy import linalg
import scipy.sparse as spa
import scipy.sparse.linalg as spla

In [ ]:
def systemInit(bond=1.43, t=-2.7):
    graphene = sisl.geom.graphene(bond)
    Ham0 = Hamiltonian(graphene)
    r = (0.1*bond, bond+1e-2)
    t = (0.0, t)
    Ham0.construct([r, t])
    return Ham0

def setup_ham_rse(Ham, tile=4, nk1=100, eta=1e-3j):
    if isinstance(tile, int): 
        Na = Nb = tile
    elif isinstance(tile, (tuple, list)):
        Na, Nb = tile
    rse = sisl.RealSpaceSE(Ham, 0, 1, (Na, Nb, 1))
    rse.setup(eta=eta, bz=sisl.MonkhorstPack(Ham, [1, nk1, 1]))
    H = Ham.tile(Na, 0).tile(Nb, 1)
    H.set_nsc([1,1,1])
    _, elec_indices = rse.real_space_coupling(ret_indices=True)
    
    all_atoms = np.arange(0, H.na)
    inside_atoms = np.delete(all_atoms, elec_indices)
    alist = np.concatenate([elec_indices, inside_atoms])
    
    H_final = H.sub(alist)
    H_final.reduce()
    
    return H_final, rse, alist, elec_indices

def dysonEQ(z, S_mat, H_mat, rse, alist):
    # if isinstance(nC, (list, tuple, np.ndarray)):
    #     nC = len(nC)
    # elif isinstance(nC, int):
    #     pass
    # else: raise ValueError(f"nC must be either (a) list of electrode indices or (b) number of electrode atoms.")
    
    invG = z*S_mat - H_mat
    RSE = rse.self_energy(z)
    
    # print(f"{                     invG.shape = }")
    # print(f"{                      RSE.shape = }")
    # print(f"{RSE[np.ix_(alist, alist)].shape = }")
    # print(f"{                    alist.shape = }")
    
    invG -= RSE[np.ix_(alist, alist)]
    return invG

def diag_inv_solver(invG):
    if spa.isspmatrix(invG):
        lu = spla.splu(invG)
        solver = lambda b: lu.solver(b)
    else:
        lu_and_piv = linalg.lu_factor(invG)
        solver = lambda b: linalg.lu_solve(lu_and_piv, b)
    return solver

def invertGF(invG, full=False):
    if full: return linalg.inv(invG)
    
    K, _ = invG.shape
    diagG = np.zeros(K, dtype=complex)
    solver = diag_inv_solver(invG)
    
    for idx in tqdm(range(K), desc="diagonal of inverse"):
        ei = np.zeros(K)
        ei[idx] = 1.0
        xi = solver(ei)
        diagG[idx] = xi[idx]
    return diagG

def calc_ldos(G):
    if G.ndim == 1: diagG = G
    elif G.ndim == 2: diagG = np.diag(G)
    else: raise ValueError(f"Don't know what went wrong, but n dims of Greens is neither 1D or 2D,\n {G.ndim = };\t{G.shape = }")
    return -(1.0/np.pi) * np.imag(diagG)

def calculate_spectral_density(energies, eta, H_dev, rse, alist, elist):
    assert isinstance(eta, complex), "Energy perturbation must be complex and should be purely imag."
    nC = len(elist) # number atoms in electrode 
    nS = len(H_dev) # number of sites/atoms
    nE = len(energies) # number of energies
    H_mat = H_dev.Hk(format="array")
    S_mat = H_dev.Sk(format="array")
    
    LDOS = np.zeros(shape=(nS, nE), dtype=float)
    for i, E in enumerate(tqdm(energies, desc="LDOS calc")):
        z = E + eta
        invG = dysonEQ(z, S_mat, H_mat, rse, alist)
        G = invertGF(invG, full=True)
        LDOS[:, i] = calc_ldos(G)
    
    return LDOS
    

In [ ]:
Na = 12
dE = 0.1
Emax = 3.0 # eV
Emin = -Emax
energies = np.arange(Emin, Emax+dE, dE)
eta = 1e-2j

# initialize TB hamiltonian
Ham = systemInit(bond=1.43, t=-2.7)

# setup Real Space Self-energies and reordered hamiltonian
H_final, rse, alist, elist = setup_ham_rse(Ham, tile=Na, nk1=100, eta=eta)

# Compute the LDOS from the Spectral Density
LDOS = calculate_spectral_density(energies, eta, H_final, rse, alist, elist)

# find number of electron indices
nC = len(elist)

# find LDOS and DOS for device ONLY
LDOS_dev = LDOS[nC:, :]
DOS_dev = LDOS_dev.sum(axis=0)


In [ ]:
data = np.load("alans_calcs.npz")
DOS_OG = data["dos"]
energies_OG = data["E"]

if not np.array_equal(energies, energies_OG):
    print("arrays not equal")
    if len(energies_OG) < len(energies):
        x = energies_OG
        xp = energies
        fp = DOS_dev
        y = np.interp(x, xp, fp)
    elif len(energies_OG) > len(energies):
        x = energies
        xp = energies_OG
        fp = DOS_OG
        y = np.interp(x, xp, fp)
else:
    print("ararys equal")
    x = energies
    xp = x
    fp = DOS_OG
    y = DOS_dev

In [ ]:

# === plot DOS ===
E_idx=31
E_idx=min(E_idx, len(energies)) -1

fig, ax = plt.subplots(1, 2, figsize=(6,4))

ax[0].plot(energies_OG, DOS_OG, c="k", label="OG method")
ax[0].plot(energies, DOS_dev, c="r", label="new method")
ax[0].vlines(x=energies[E_idx],ymin=0,ymax=600,color='k',linestyle='dashed')


for a in ax:
    a.set_xlabel("Energy (eV)")
    a.set_ylabel("DOS (states / eV)")
    a.set(ylim=(0,100), xlim=(-3,3))
    a.legend()

ax[1].plot(x, np.abs(fp - y), label="Diff")
None

In [ ]:
bond = 1.43
gr = sisl.geom.graphene(bond=bond)
Ham0 = Hamiltonian(gr)
r = (0.1*bond, bond+1e-2)
t = (0.0, -2.7)
Ham0.construct([r, t])
Ham = Ham0.tile(1,0).tile(1,1)

Na = Nb = 12
eta = 1e-2j
nk1 = int(np.ceil(3*900/Nb))

rse = sisl.RealSpaceSE(Ham, 0, 1, (Na, Nb, 1))
rse.setup(eta=eta, bz=sisl.MonkhorstPack(Ham, [1, nk1, 1]))
HamNN = Ham.tile(Na, 0).tile(Nb, 1)
geomNN = HamNN.geometry
Ham_elec, elec_indices = rse.real_space_coupling(ret_indices=True)
nC = len(elec_indices)
HamNN.set_nsc([1,1,1])

all_atoms = np.arange(0, HamNN.na)
inside_atoms = np.delete(all_atoms, elec_indices, axis=None)
alist = np.concatenate([elec_indices, inside_atoms])

HamNN_reordered = HamNN.sub(alist)
HamNN_reordered.reduce()
H_sub = HamNN_reordered.Hk(format="array")
S_sub = HamNN_reordered.Sk(format="array")

In [ ]:
dE = 0.1
Emax = 3.0 # eV
Emin = -Emax
energies = np.arange(Emin, Emax+dE, dE)

nE = len(energies)
N = len(HamNN_reordered)

LDOS = np.empty(shape=(nE, N), dtype=float)


for idx, E in enumerate(tqdm(energies, desc="LDOS")):
    z = E + eta
    invG = z*S_sub - H_sub
    RSE = rse.self_energy(z)
    RSE_reordered = RSE[np.ix_(alist, alist)]
    invG[0:len(alist), 0:len(alist)] -= RSE_reordered
    G = np.linalg.inv(invG)
    diagG = np.diag(G)
    LDOS[idx, :] = -(1.0/np.pi) * np.imag(diagG)
    
LDOS_dev = LDOS[nC:, :]
DOS_dev = LDOS_dev.sum(axis=1)

In [ ]:
DATA = np.load("alans_calcs.npz")
DOS_OG = DATA["dos"]

In [ ]:
# === plot DOS ===
E_idx=31
E_idx=min(E_idx, len(energies)) -1

fig, ax = plt.subplots(1, 2, figsize=(6,4))

ax[0].plot(energies, DOS_OG, c="k", label="OG method")
ax[0].plot(energies, DOS_dev, c="r", label="new method")
ax[0].vlines(x=energies[E_idx],ymin=0,ymax=600,color='k',linestyle='dashed')


for a in ax:
    a.set_xlabel("Energy (eV)")
    a.set_ylabel("DOS (states / eV)")
    a.set(ylim=(0,100), xlim=(-3,3))
    a.legend()

ax[1].plot(energies, np.abs(DOS_OG - DOS_dev), label="Diff")
fig.tight_layout()
None